In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import KFold, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, PolynomialFeatures, SplineTransformer
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, f1_score, make_scorer

from sklearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

from sklearn.experimental import enable_iterative_imputer  # Mandatory line
from sklearn.impute import IterativeImputer

RANDOM_STATE = 51026

In [2]:
train = pd.read_csv(
    "QSAR_challenge_train_80pct.csv"
)

submission_data = pd.read_csv(
    "QSAR_challenge_test_20pct_NO_TARGETS.csv"
)

print("Labelled dataset:", train.shape)
print("Submission dataset:", submission_data.shape)

Labelled dataset: (623, 12)
Submission dataset: (156, 10)


In [3]:
feature_cols = [
    "nHM",
    "piPC09",
    "PCD",
    "X2Av",
    "MLOGP",
    "ON1V",
    "N-072",
    "B02[C-N]",
    "F04[C-O]"
]

X = train[feature_cols].copy()
y = train["regression_target"].copy()

In [4]:
X_submission = submission_data[feature_cols].copy()

In [5]:
cv_reg = KFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [6]:
def evaluate_model(model, X, y, cv):

    results = []

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X),
        start=1
    ):

        X_train_fold = X.iloc[train_idx]
        X_val_fold = X.iloc[val_idx]

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        model.fit(
            X_train_fold,
            y_train_fold
        )

        y_pred = model.predict(
            X_val_fold
        ).ravel()

        results.append({
            "fold": fold,
            "MAE": mean_absolute_error(
                y_val_fold,
                y_pred
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_val_fold,
                    y_pred
                )
            ),
            "R2": r2_score(
                y_val_fold,
                y_pred
            )
        })

    results = pd.DataFrame(results)

    summary = {
        "MAE_mean": results["MAE"].mean(),
        "MAE_std": results["MAE"].std(),
        "RMSE_mean": results["RMSE"].mean(),
        "RMSE_std": results["RMSE"].std(),
        "R2_mean": results["R2"].mean(),
        "R2_std": results["R2"].std()
    }

    return results, summary

In [7]:
from sklearn.cross_decomposition import PLSRegression
from sklearn.decomposition import PCA

In [60]:
pls_results = []

for n_components in range(1, 10):

    model = Pipeline([
        ("imputer", IterativeImputer(
            random_state=RANDOM_STATE
        )),
        ("scaler", StandardScaler()),
        ("pls", PLSRegression(
            n_components=n_components,
            max_iter=2000
        ))
    ])

    _, summary = evaluate_model(
        model,
        X,
        y,
        cv_reg
    )

    pls_results.append({
        "model": "PLS",
        "components": n_components,
        **summary
    })

pls_results_df = pd.DataFrame(pls_results)

pls_results_df.sort_values(
    "R2_mean",
    ascending=False
)

,model,components,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
4,PLS,5,0.632079,0.038651,0.826784,0.055145,0.603649,0.081793
5,PLS,6,0.632096,0.038953,0.826869,0.058089,0.603045,0.086830
8,PLS,9,0.633246,0.036743,0.827426,0.057333,0.602528,0.086660
6,PLS,7,0.632877,0.038036,0.827347,0.058012,0.602526,0.087431
7,PLS,8,0.633218,0.036756,0.827426,0.057358,0.602524,0.086695
3,PLS,4,0.631737,0.039575,0.827469,0.057997,0.602521,0.086237
2,PLS,3,0.634659,0.038185,0.831634,0.051125,0.599534,0.076555
1,PLS,2,0.645906,0.041397,0.836956,0.058022,0.593881,0.083384
0,PLS,1,0.655378,0.043490,0.850727,0.062880,0.579090,0.097869


In [ ]:
ridge_results = []

for alpha in [
    0.01,
    0.1,
    1,
    3,
    10,
    30,
    100,
    300,
    1000
]:

    model = Pipeline([
        ("imputer", IterativeImputer(
            random_state=RANDOM_STATE
        )),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(
            alpha=alpha
        ))
    ])

    _, summary = evaluate_model(
        model,
        X,
        y,
        cv_reg
    )

    ridge_results.append({
        "model": "Ridge",
        "alpha": alpha,
        **summary
    })

ridge_results_df = pd.DataFrame(ridge_results)

ridge_results_df.sort_values(
    "R2_mean",
    ascending=False
)

In [61]:
poly_results = []

for degree in [2, 3]:

    for alpha in [
        0.1,
        1,
        10,
        30,
        100,
        300,
        1000
    ]:

        model = Pipeline([
            ("imputer", IterativeImputer(
                random_state=RANDOM_STATE
            )),

            ("poly", PolynomialFeatures(
                degree=degree,
                include_bias=False
            )),

            ("scaler", StandardScaler()),

            ("ridge", Ridge(
                alpha=alpha
            ))
        ])

        _, summary = evaluate_model(
            model,
            X,
            y,
            cv_reg
        )

        poly_results.append({
            "model": "Polynomial_Ridge",
            "degree": degree,
            "alpha": alpha,
            **summary
        })

poly_results_df = pd.DataFrame(
    poly_results
)

poly_results_df.sort_values(
    "R2_mean",
    ascending=False
).head(10)

,model,degree,alpha,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
2,Polynomial_Ridge,2,10.0,0.602192,0.043248,0.801247,0.050278,0.630190,0.054206
1,Polynomial_Ridge,2,1.0,0.598345,0.049884,0.805540,0.060094,0.626646,0.054603
3,Polynomial_Ridge,2,30.0,0.612532,0.042487,0.811430,0.048775,0.619920,0.062453
0,Polynomial_Ridge,2,0.1,0.601075,0.053908,0.815846,0.066368,0.616870,0.059759
4,Polynomial_Ridge,2,100.0,0.633364,0.039757,0.828446,0.046608,0.603210,0.069777
9,Polynomial_Ridge,3,10.0,0.608217,0.061629,0.831229,0.065477,0.601708,0.064621
10,Polynomial_Ridge,3,30.0,0.614121,0.056901,0.835517,0.066795,0.597195,0.069054
11,Polynomial_Ridge,3,100.0,0.631385,0.046665,0.843275,0.060472,0.589449,0.069862
5,Polynomial_Ridge,2,300.0,0.661424,0.033203,0.848251,0.040255,0.584492,0.066966
8,Polynomial_Ridge,3,1.0,0.614782,0.068624,0.855084,0.104057,0.577044,0.091927


In [ ]:
spline_results = []

for n_knots in [4, 5, 6, 7, 8]:

    for degree in [2, 3]:

        for alpha in [
            1,
            10,
            30,
            100,
            300,
            1000
        ]:

            model = Pipeline([
                ("imputer", IterativeImputer(
                    random_state=RANDOM_STATE
                )),

                ("spline", SplineTransformer(
                    n_knots=n_knots,
                    degree=degree,
                    include_bias=False
                )),

                ("scaler", StandardScaler()),

                ("ridge", Ridge(
                    alpha=alpha
                ))
            ])

            _, summary = evaluate_model(
                model,
                X,
                y,
                cv_reg
            )

            spline_results.append({
                "model": "Spline_Ridge",
                "knots": n_knots,
                "degree": degree,
                "alpha": alpha,
                **summary
            })

spline_results_df = pd.DataFrame(
    spline_results
)

spline_results_df.sort_values(
    "R2_mean",
    ascending=False
).head(15)

In [ ]:
pcr_results = []

for n_components in range(1, 10):

    model = Pipeline([
        ("imputer", IterativeImputer(
            random_state=RANDOM_STATE
        )),

        ("scaler", StandardScaler()),

        ("pca", PCA(
            n_components=n_components
        )),

        ("ridge", Ridge(
            alpha=1
        ))
    ])

    _, summary = evaluate_model(
        model,
        X,
        y,
        cv_reg
    )

    pcr_results.append({
        "model": "PCR",
        "components": n_components,
        **summary
    })

pcr_results_df = pd.DataFrame(
    pcr_results
)

pcr_results_df.sort_values(
    "R2_mean",
    ascending=False
)

In [ ]:
all_results = pd.concat(
    [
        pls_results_df,
        ridge_results_df,
        poly_results_df,
        spline_results_df,
        pcr_results_df
    ],
    ignore_index=True,
    sort=False
)

In [ ]:
all_results.sort_values(
    "R2_mean",
    ascending=False
).head(20)

In [ ]:
all_results.sort_values(
    "MAE_mean"
).head(20)

In [ ]:
pls_results = []

for n_components in range(1, 10):

    model = Pipeline([
        ("imputer", IterativeImputer(
            random_state=RANDOM_STATE
        )),
        ("scaler", StandardScaler()),
        ("pls", PLSRegression(
            n_components=n_components,
            max_iter=2000
        ))
    ])

    _, summary = evaluate_model(
        model,
        X,
        y,
        cv_reg
    )

    pls_results.append({
        "model": "PLS",
        "components": n_components,
        **summary
    })

pls_results_df = pd.DataFrame(pls_results)

pls_results_df.sort_values(
    "R2_mean",
    ascending=False
)

In [ ]:
corr = X.corr()

corr.round(3)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 8))

plt.imshow(corr, interpolation="nearest")
plt.colorbar()

plt.xticks(
    range(len(corr.columns)),
    corr.columns,
    rotation=90
)

plt.yticks(
    range(len(corr.columns)),
    corr.columns
)

plt.title("Feature correlation matrix")
plt.tight_layout()
plt.show()

In [ ]:
target_corr = pd.concat(
    [X, y.rename("regression_target")],
    axis=1
).corr()["regression_target"].drop("regression_target")

target_corr.sort_values(
    key=abs,
    ascending=False
).round(3)

In [ ]:
from sklearn.base import clone

def evaluate_feature_selection_model(
    model,
    X,
    y,
    cv,
    n_features
):

    results = []

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X),
        start=1
    ):

        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # Feature selection using ONLY the training fold
        correlations = X_train_fold.copy()
        correlations["target"] = y_train_fold.values

        target_corr = (
            correlations
            .corr()["target"]
            .drop("target")
            .abs()
            .sort_values(ascending=False)
        )

        selected_features = target_corr.head(
            n_features
        ).index.tolist()

        X_train_selected = X_train_fold[
            selected_features
        ]

        X_val_selected = X_val_fold[
            selected_features
        ]

        fitted_model = clone(model)

        fitted_model.fit(
            X_train_selected,
            y_train_fold
        )

        y_pred = fitted_model.predict(
            X_val_selected
        ).ravel()

        results.append({
            "fold": fold,
            "n_features": n_features,
            "features": selected_features,
            "MAE": mean_absolute_error(
                y_val_fold,
                y_pred
            ),
            "RMSE": np.sqrt(
                mean_squared_error(
                    y_val_fold,
                    y_pred
                )
            ),
            "R2": r2_score(
                y_val_fold,
                y_pred
            )
        })

    results = pd.DataFrame(results)

    summary = {
        "MAE_mean": results["MAE"].mean(),
        "MAE_std": results["MAE"].std(),
        "RMSE_mean": results["RMSE"].mean(),
        "RMSE_std": results["RMSE"].std(),
        "R2_mean": results["R2"].mean(),
        "R2_std": results["R2"].std()
    }

    return results, summary

In [ ]:
spline_ridge = Pipeline([

    ("imputer", IterativeImputer(
        random_state=RANDOM_STATE
    )),

    ("spline", SplineTransformer(
        n_knots=6,
        degree=3,
        include_bias=False
    )),

    ("scaler", StandardScaler()),

    ("model", Ridge(alpha=100))
])


pls5 = Pipeline([

    ("imputer", IterativeImputer(
        random_state=RANDOM_STATE
    )),

    ("scaler", StandardScaler()),

    ("model", PLSRegression(
        n_components=5,
        max_iter=2000
    ))
])

In [ ]:
feature_selection_results = []

for model_name, model in [
    ("Spline_Ridge", spline_ridge),
    ("PLS5", pls5)
]:

    for n_features in range(5, 10):

        _, summary = evaluate_feature_selection_model(
            model=model,
            X=X,
            y=y,
            cv=cv_reg,
            n_features=n_features
        )

        feature_selection_results.append({
            "model": model_name,
            "n_features": n_features,
            **summary
        })

feature_selection_df = pd.DataFrame(
    feature_selection_results
)

feature_selection_df.sort_values(
    ["model", "R2_mean"],
    ascending=[True, False]
)

In [ ]:
from sklearn.impute import SimpleImputer

spline_variants = {

    "Spline_median": Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("spline", SplineTransformer(
            n_knots=6,
            degree=3,
            include_bias=False
        )),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=100))
    ]),

    "Spline_mean": Pipeline([
        ("imputer", SimpleImputer(strategy="mean")),
        ("spline", SplineTransformer(
            n_knots=6,
            degree=3,
            include_bias=False
        )),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=100))
    ]),

    "Spline_iterative": Pipeline([
        ("imputer", IterativeImputer(
            random_state=RANDOM_STATE
        )),
        ("spline", SplineTransformer(
            n_knots=6,
            degree=3,
            include_bias=False
        )),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=100))
    ])
}

In [ ]:
imputation_results = []

for model_name, model in spline_variants.items():

    _, summary = evaluate_model(
        model,
        X,
        y,
        cv_reg
    )

    imputation_results.append({
        "model": model_name,
        **summary
    })

imputation_results_df = pd.DataFrame(
    imputation_results
)

imputation_results_df.sort_values(
    "R2_mean",
    ascending=False
)


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))

plt.scatter(
    X["MLOGP"],
    y,
    alpha=0.6
)

plt.xlabel("MLOGP")
plt.ylabel("Regression target")
plt.title("MLOGP vs regression target")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import math

top_features = [
    "nHM",
    "piPC09",
    "PCD",
    "X2Av",
    "MLOGP",
    "ON1V",
    "N-072",
    "B02[C-N]",
    "F04[C-O]"]

n_features = len(top_features)
n_cols = 4
n_rows = math.ceil(n_features / n_cols)

# Create grid dynamically based on feature count
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 3.8 * n_rows))

# Flatten axes array for easy indexing (handles single-row vs multi-row cases)
axes = axes.flatten() if n_features > 1 else [axes]

for i, feature in enumerate(top_features):
    axes[i].scatter(X[feature], y, alpha=0.6, color="#1f77b4")
    axes[i].set_xlabel(feature, fontweight="bold")
    axes[i].set_ylabel("Regression Target", fontweight="bold")
    axes[i].set_title(f"{feature} vs target", fontsize=10)
    axes[i].grid(True, linestyle=":", alpha=0.5)

# Hide any empty subplot frames in the last row
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

Mixed preprocessing

In [9]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.linear_model import Ridge

In [10]:
spline_features = [
    "MLOGP",
    "piPC09",
    "X2Av",
    "ON1V"
]

linear_features = [
    "nHM",
    "PCD",
    "N-072",
    "B02[C-N]",
    "F04[C-O]"
]

mixed_preprocessor = ColumnTransformer([
    
    (
        "spline",
        Pipeline([
            ("imputer", IterativeImputer(
                random_state=RANDOM_STATE
            )),
            ("spline", SplineTransformer(
                n_knots=6,
                degree=3,
                include_bias=False
            ))
        ]),
        spline_features
    ),

    (
        "linear",
        Pipeline([
            ("imputer", IterativeImputer(
                random_state=RANDOM_STATE
            ))
        ]),
        linear_features
    )
])



In [12]:
mixed_spline_ridge = Pipeline([
    ("preprocessor", mixed_preprocessor),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=100))
])

In [13]:
_, mixed_summary = evaluate_model(
    mixed_spline_ridge,
    X,
    y,
    cv_reg
)

mixed_summary

{'MAE_mean': np.float64(0.6172838425085962),
 'MAE_std': np.float64(0.04462848870941727),
 'RMSE_mean': np.float64(0.8009571446057964),
 'RMSE_std': np.float64(0.07106916891293763),
 'R2_mean': np.float64(0.6279795099087173),
 'R2_std': np.float64(0.07983287161802839)}

In [ ]:
for feature in feature_cols:

    values = X[feature].dropna()

    print(
        f"\n{feature}"
    )
    print(
        "n_unique:",
        values.nunique()
    )
    print(
        "unique:",
        np.sort(values.unique())[:30]
    )

In [ ]:
X_pi_indicator = X.copy()

X_pi_indicator["piPC09_zero"] = (
    X_pi_indicator["piPC09"] == 0
).astype(int)

spline_ridge_pi_indicator = Pipeline([

    ("imputer", IterativeImputer(
        random_state=RANDOM_STATE
    )),

    ("spline", SplineTransformer(
        n_knots=6,
        degree=3,
        include_bias=False
    )),

    ("scaler", StandardScaler()),

    ("model", Ridge(alpha=100))
])

In [ ]:
_, pi_indicator_summary = evaluate_model(
    spline_ridge_pi_indicator,
    X_pi_indicator,
    y,
    cv_reg
)

pi_indicator_summary

In [ ]:
print("Current:")
print({
    "MAE": 0.599932,
    "RMSE": 0.778609,
    "R2": 0.646830
})

print("\nWith piPC09_zero:")
print(pi_indicator_summary)

Plain GAM

GAM with MLOGP upweighted

In [14]:
from pygam import LinearGAM, s, l

In [15]:
gam_plain = LinearGAM(
    s(0) +      # nHM
    s(1) +      # piPC09
    s(2) +      # PCD
    s(3) +      # X2Av
    s(4) +      # MLOGP
    s(5) +      # ON1V
    s(6) +      # N-072
    l(7) +      # B02[C-N]
    l(8)        # F04[C-O]
)

def evaluate_gam(gam, X, y, cv):
    results = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X), start=1):

        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # Imputation fitted only on training fold
        imputer = IterativeImputer(random_state=RANDOM_STATE)

        X_train_imp = imputer.fit_transform(X_train_fold)
        X_val_imp = imputer.transform(X_val_fold)

        # Fit GAM only on training fold
        gam.fit(X_train_imp, y_train_fold)

        y_pred = gam.predict(X_val_imp)

        results.append({
            "fold": fold,
            "MAE": mean_absolute_error(y_val_fold, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_val_fold, y_pred)),
            "R2": r2_score(y_val_fold, y_pred)
        })

    results = pd.DataFrame(results)

    summary = {
        "MAE_mean": results["MAE"].mean(),
        "MAE_std": results["MAE"].std(),
        "RMSE_mean": results["RMSE"].mean(),
        "RMSE_std": results["RMSE"].std(),
        "R2_mean": results["R2"].mean(),
        "R2_std": results["R2"].std()
    }

    return results, summary

In [16]:
gam_plain_results, gam_plain_summary = evaluate_gam(
    gam_plain,
    X,
    y,
    cv_reg
)

gam_plain_results

,fold,MAE,RMSE,R2
0,1,0.687339,0.899164,0.568860
1,2,0.626819,0.842218,0.523970
2,3,0.525771,0.665203,0.751265
3,4,0.591367,0.791076,0.663201
4,5,0.596955,0.788732,0.645581


In [68]:
gam_plain_summary

{'MAE_mean': np.float64(0.6056503721613358),
 'MAE_std': np.float64(0.05869722910532163),
 'RMSE_mean': np.float64(0.7972785489809856),
 'RMSE_std': np.float64(0.08653856788708464),
 'R2_mean': np.float64(0.6305753122384853),
 'R2_std': np.float64(0.08807517548882436)}

In [17]:
gam_mlogp_weighted = LinearGAM(
    s(0, n_splines=10) +   # nHM
    s(1, n_splines=10) +   # piPC09
    s(2, n_splines=10) +   # PCD
    s(3, n_splines=10) +   # X2Av
    s(4, n_splines=20) +   # MLOGP - more flexible
    s(5, n_splines=10) +   # ON1V
    s(6, n_splines=10) +   # N-072
    l(7) +                  # B02[C-N]
    l(8)                    # F04[C-O]
)

In [18]:
gam_mlogp_results, gam_mlogp_summary = evaluate_gam(
    gam_mlogp_weighted,
    X,
    y,
    cv_reg
)

gam_mlogp_results

,fold,MAE,RMSE,R2
0,1,0.706155,0.952133,0.516567
1,2,0.621375,0.846746,0.518838
2,3,0.524354,0.674692,0.744118
3,4,0.589715,0.772723,0.678648
4,5,0.585692,0.745735,0.683169


In [19]:
gam_mlogp_summary

{'MAE_mean': np.float64(0.6054579204650029),
 'MAE_std': np.float64(0.06635064389660497),
 'RMSE_mean': np.float64(0.7984058719131509),
 'RMSE_std': np.float64(0.10572108609588585),
 'R2_mean': np.float64(0.6282677824664555),
 'R2_std': np.float64(0.10419379910759159)}

MLOGP-focused GAM

In [20]:
gam_mlogp_focused = LinearGAM(
    l(0) +                  # nHM
    l(1) +                  # piPC09
    l(2) +                  # PCD
    l(3) +                  # X2Av
    s(4, n_splines=10) +    # MLOGP - nonlinear
    l(5) +                  # ON1V
    l(6) +                  # N-072
    l(7) +                  # B02[C-N]
    l(8)                    # F04[C-O]
)

gam_mlogp_focused_results, gam_mlogp_focused_summary = evaluate_gam(
    gam_mlogp_focused,
    X,
    y,
    cv_reg
)

In [21]:
gam_mlogp_focused_results

,fold,MAE,RMSE,R2
0,1,0.651798,0.854479,0.610647
1,2,0.661671,0.890935,0.467306
2,3,0.565867,0.754065,0.680371
3,4,0.616259,0.784007,0.669194
4,5,0.596509,0.760308,0.670665


In [22]:
gam_mlogp_focused_summary

{'MAE_mean': np.float64(0.6184209623219936),
 'MAE_std': np.float64(0.03946952366185574),
 'RMSE_mean': np.float64(0.8087588883399471),
 'RMSE_std': np.float64(0.06081716996694548),
 'R2_mean': np.float64(0.6196364749827786),
 'R2_std': np.float64(0.08948998100051637)}

In [23]:
gam_comparison = pd.DataFrame({
    "Plain_GAM": gam_plain_summary,
    "MLOGP_focused": gam_mlogp_focused_summary,
    "Spline_Ridge": {
        "MAE_mean": 0.599932,
        "MAE_std": 0.0,       # placeholder; don't use for comparison
        "RMSE_mean": 0.778609,
        "RMSE_std": 0.0,
        "R2_mean": 0.646829,
        "R2_std": 0.089537
    }
})

gam_comparison

,Plain_GAM,MLOGP_focused,Spline_Ridge
MAE_mean,0.605650,0.618421,0.599932
MAE_std,0.058697,0.039470,0.000000
RMSE_mean,0.797279,0.808759,0.778609
RMSE_std,0.086539,0.060817,0.000000
R2_mean,0.630575,0.619636,0.646829
R2_std,0.088075,0.089490,0.089537


Weighted GAM

In [24]:
gam_weighted = LinearGAM(
    s(0) +      # nHM
    s(1) +      # piPC09
    s(2) +      # PCD
    s(3) +      # X2Av
    s(4) +      # MLOGP
    s(5) +      # ON1V
    s(6) +      # N-072
    l(7) +      # B02[C-N]
    l(8)        # F04[C-O]
)

def evaluate_weighted_gam(gam, X, y, cv, weight_strength=1.0):
    results = []

    for fold, (train_idx, val_idx) in enumerate(cv.split(X), start=1):

        X_train_fold = X.iloc[train_idx].copy()
        X_val_fold = X.iloc[val_idx].copy()

        y_train_fold = y.iloc[train_idx]
        y_val_fold = y.iloc[val_idx]

        # Imputation fitted only on training fold
        imputer = IterativeImputer(random_state=RANDOM_STATE)

        X_train_imp = imputer.fit_transform(X_train_fold)
        X_val_imp = imputer.transform(X_val_fold)

        # Robust target scale calculated only from training data
        median_y = y_train_fold.median()
        mad_y = np.median(
            np.abs(y_train_fold - median_y)
        )

        # Avoid division by zero
        if mad_y == 0:
            mad_y = 1.0

        # Distance from the target center
        target_distance = (
            np.abs(y_train_fold - median_y) / mad_y
        )

        # Give more weight to observations farther from the center
        weights = 1 + weight_strength * target_distance

        # Fit weighted GAM
        gam.fit(
            X_train_imp,
            y_train_fold,
            weights=weights
        )

        y_pred = gam.predict(X_val_imp)

        results.append({
            "fold": fold,
            "MAE": mean_absolute_error(y_val_fold, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_val_fold, y_pred)),
            "R2": r2_score(y_val_fold, y_pred)
        })

    results = pd.DataFrame(results)

    summary = {
        "MAE_mean": results["MAE"].mean(),
        "MAE_std": results["MAE"].std(),
        "RMSE_mean": results["RMSE"].mean(),
        "RMSE_std": results["RMSE"].std(),
        "R2_mean": results["R2"].mean(),
        "R2_std": results["R2"].std()
    }

    return results, summary

In [25]:
gam_weighted_results, gam_weighted_summary = evaluate_weighted_gam(
    gam_weighted,
    X,
    y,
    cv_reg,
    weight_strength=1.0
)

In [26]:
gam_weighted_results

,fold,MAE,RMSE,R2
0,1,0.732418,0.997142,0.469782
1,2,0.683583,0.900864,0.455367
2,3,0.535419,0.666317,0.750431
3,4,0.596950,0.783834,0.669340
4,5,0.601294,0.792545,0.642145


Spline +Rigde with interactions

In [27]:
import numpy as np
import pandas as pd

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import SplineTransformer, StandardScaler
from sklearn.impute import IterativeImputer
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline

class SplineInteractions(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        n_knots=6,
        degree=3,
        interaction_with=4
    ):
        self.n_knots = n_knots
        self.degree = degree
        self.interaction_with = interaction_with

    def fit(self, X, y=None):

        X = np.asarray(X)

        self.n_features_in_ = X.shape[1]

        self.splines_ = []

        for j in range(self.n_features_in_):
            spline = SplineTransformer(
                n_knots=self.n_knots,
                degree=self.degree,
                include_bias=False
            )

            spline.fit(X[:, [j]])
            self.splines_.append(spline)

        return self

    def transform(self, X):

        X = np.asarray(X)

        # Transform every variable separately
        spline_features = []

        for j, spline in enumerate(self.splines_):
            Z_j = spline.transform(X[:, [j]])
            spline_features.append(Z_j)

        # Main spline effects
        output = spline_features.copy()

        # Add interactions involving MLOGP
        mlogp_basis = spline_features[self.interaction_with]

        for j in range(self.n_features_in_):

            if j == self.interaction_with:
                continue

            interaction = (
                mlogp_basis[:, :, None]
                *
                spline_features[j][:, None, :]
            )

            interaction = interaction.reshape(
                interaction.shape[0],
                -1
            )

            output.append(interaction)

        return np.hstack(output)

spline_interaction_ridge = Pipeline([
    (
        "imputer",
        IterativeImputer(
            random_state=RANDOM_STATE
        )
    ),

    (
        "features",
        SplineInteractions(
            n_knots=6,
            degree=3,
            interaction_with=4
        )
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        Ridge(alpha=100)
    )
])

In [28]:
interaction_results, interaction_summary = evaluate_model(
    spline_interaction_ridge,
    X,
    y,
    cv_reg
)

interaction_results

,fold,MAE,RMSE,R2
0,1,38.817099,351.694691,-65957.736927
1,2,1.297608,7.346882,-35.223585
2,3,0.538257,0.838018,0.605237
3,4,0.567316,0.766210,0.684042
4,5,4.347451,39.234777,-876.002742


In [29]:
interaction_summary

{'MAE_mean': np.float64(9.113546358618475),
 'MAE_std': np.float64(16.678423294955675),
 'RMSE_mean': np.float64(79.97611568655871),
 'RMSE_std': np.float64(152.72761223728483),
 'R2_mean': np.float64(-13373.534795172473),
 'R2_std': np.float64(29397.850764475515)}

Now using not the full numberof interactions, but only MLPGP * n

In [30]:
class SplinePlusMLOGPInteractions(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        n_knots=6,
        degree=3,
        mlogp_index=4
    ):
        self.n_knots = n_knots
        self.degree = degree
        self.mlogp_index = mlogp_index

    def fit(self, X, y=None):

        X = np.asarray(X)

        self.n_features_in_ = X.shape[1]

        # Fit one spline transformer per original feature
        self.splines_ = []

        for j in range(self.n_features_in_):

            spline = SplineTransformer(
                n_knots=self.n_knots,
                degree=self.degree,
                include_bias=False
            )

            spline.fit(X[:, [j]])
            self.splines_.append(spline)

        return self

    def transform(self, X):

        X = np.asarray(X)

        # Original spline features
        spline_features = []

        for j, spline in enumerate(self.splines_):
            Z_j = spline.transform(X[:, [j]])
            spline_features.append(Z_j)

        output = spline_features.copy()

        # Raw MLOGP interactions
        mlogp = X[:, self.mlogp_index]

        for j in range(self.n_features_in_):

            if j == self.mlogp_index:
                continue

            interaction = (
                mlogp * X[:, j]
            ).reshape(-1, 1)

            output.append(interaction)

        return np.hstack(output)

spline_mlogp_interactions = Pipeline([
    (
        "imputer",
        IterativeImputer(
            random_state=RANDOM_STATE
        )
    ),

    (
        "features",
        SplinePlusMLOGPInteractions(
            n_knots=6,
            degree=3,
            mlogp_index=4
        )
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        Ridge(alpha=100)
    )
])

mlogp_interaction_results, mlogp_interaction_summary = evaluate_model(
    spline_mlogp_interactions,
    X,
    y,
    cv_reg
)

In [31]:
mlogp_interaction_results

,fold,MAE,RMSE,R2
0,1,0.659764,0.849135,0.615502
1,2,0.631125,0.847372,0.518126
2,3,0.516431,0.641089,0.768971
3,4,0.585034,0.773345,0.678130
4,5,0.598865,0.759941,0.670983


In [32]:
mlogp_interaction_summary

{'MAE_mean': np.float64(0.5982437420092621),
 'MAE_std': np.float64(0.054162031560746865),
 'RMSE_mean': np.float64(0.7741764512395123),
 'RMSE_std': np.float64(0.08498813680133326),
 'R2_mean': np.float64(0.6503424106691762),
 'R2_std': np.float64(0.09214928404642314)}

Improvement!!!

Which interactions are the most meaningful?

Adding interactions one by one

In [33]:
class SplinePlusOneInteraction(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        n_knots=6,
        degree=3,
        mlogp_index=4,
        interaction_index=0
    ):
        self.n_knots = n_knots
        self.degree = degree
        self.mlogp_index = mlogp_index
        self.interaction_index = interaction_index

    def fit(self, X, y=None):

        X = np.asarray(X)

        self.n_features_in_ = X.shape[1]

        self.splines_ = []

        for j in range(self.n_features_in_):

            spline = SplineTransformer(
                n_knots=self.n_knots,
                degree=self.degree,
                include_bias=False
            )

            spline.fit(X[:, [j]])
            self.splines_.append(spline)

        return self

    def transform(self, X):

        X = np.asarray(X)

        # Original spline features
        spline_features = []

        for j, spline in enumerate(self.splines_):
            Z_j = spline.transform(X[:, [j]])
            spline_features.append(Z_j)

        output = spline_features.copy()

        # One MLOGP interaction
        mlogp = X[:, self.mlogp_index]
        other = X[:, self.interaction_index]

        interaction = (
            mlogp * other
        ).reshape(-1, 1)

        output.append(interaction)

        return np.hstack(output)

def evaluate_single_mlogp_interaction(
    interaction_index,
    X,
    y,
    cv
):

    model = Pipeline([
        (
            "imputer",
            IterativeImputer(
                random_state=RANDOM_STATE
            )
        ),

        (
            "features",
            SplinePlusOneInteraction(
                n_knots=6,
                degree=3,
                mlogp_index=4,
                interaction_index=interaction_index
            )
        ),

        (
            "scaler",
            StandardScaler()
        ),

        (
            "model",
            Ridge(alpha=100)
        )
    ])

    _, summary = evaluate_model(
        model,
        X,
        y,
        cv
    )

    return summary



interaction_features = {
    0: "MLOGP × nHM",
    1: "MLOGP × piPC09",
    2: "MLOGP × PCD",
    3: "MLOGP × X2Av",
    5: "MLOGP × ON1V",
    6: "MLOGP × N-072",
    7: "MLOGP × B02[C-N]",
    8: "MLOGP × F04[C-O]"
}

interaction_results = []

for feature_index, interaction_name in interaction_features.items():

    summary = evaluate_single_mlogp_interaction(
        feature_index,
        X,
        y,
        cv_reg
    )

    interaction_results.append({
        "interaction": interaction_name,
        **summary
    })

interaction_results = pd.DataFrame(interaction_results)

In [34]:
interaction_results.shape

(8, 7)

In [35]:
interaction_results.sort_values(
    "R2_mean",
    ascending=False
)

,interaction,MAE_mean,MAE_std,RMSE_mean,RMSE_std,R2_mean,R2_std
7,MLOGP × F04[C-O],0.595419,0.053397,0.771205,0.083234,0.653418,0.088393
0,MLOGP × nHM,0.599471,0.054187,0.776657,0.083773,0.648224,0.092195
6,MLOGP × B02[C-N],0.599173,0.052962,0.778013,0.082226,0.647339,0.089729
5,MLOGP × N-072,0.600378,0.053017,0.778915,0.081795,0.646482,0.090118
1,MLOGP × piPC09,0.600664,0.052945,0.779881,0.081643,0.645646,0.090132
4,MLOGP × ON1V,0.600669,0.052127,0.780301,0.079596,0.645450,0.088669
3,MLOGP × X2Av,0.602212,0.051077,0.780158,0.081321,0.645288,0.090682
2,MLOGP × PCD,0.600884,0.053844,0.780389,0.082507,0.645117,0.090637


In [36]:
comparison = pd.concat([
    pd.DataFrame([{
        "interaction": "Baseline Spline Ridge",
        "MAE_mean": 0.599932,
        "RMSE_mean": 0.778609,
        "R2_mean": 0.646829
    }]),
    interaction_results[
        ["interaction", "MAE_mean", "RMSE_mean", "R2_mean"]
    ]
], ignore_index=True)

comparison.sort_values(
    "R2_mean",
    ascending=False
)

,interaction,MAE_mean,RMSE_mean,R2_mean
8,MLOGP × F04[C-O],0.595419,0.771205,0.653418
1,MLOGP × nHM,0.599471,0.776657,0.648224
7,MLOGP × B02[C-N],0.599173,0.778013,0.647339
0,Baseline Spline Ridge,0.599932,0.778609,0.646829
6,MLOGP × N-072,0.600378,0.778915,0.646482
2,MLOGP × piPC09,0.600664,0.779881,0.645646
5,MLOGP × ON1V,0.600669,0.780301,0.645450
4,MLOGP × X2Av,0.602212,0.780158,0.645288
3,MLOGP × PCD,0.600884,0.780389,0.645117


add F04[C-O] + nHM

In [37]:
class SplinePlusSelectedInteractions(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        n_knots=6,
        degree=3,
        mlogp_index=4,
        interaction_indices=None
    ):
        self.n_knots = n_knots
        self.degree = degree
        self.mlogp_index = mlogp_index
        self.interaction_indices = interaction_indices

    def fit(self, X, y=None):

        X = np.asarray(X)

        self.n_features_in_ = X.shape[1]
        self.splines_ = []

        for j in range(self.n_features_in_):

            spline = SplineTransformer(
                n_knots=self.n_knots,
                degree=self.degree,
                include_bias=False
            )

            spline.fit(X[:, [j]])
            self.splines_.append(spline)

        return self

    def transform(self, X):

        X = np.asarray(X)

        spline_features = []

        for j, spline in enumerate(self.splines_):
            Z_j = spline.transform(X[:, [j]])
            spline_features.append(Z_j)

        output = spline_features.copy()

        mlogp = X[:, self.mlogp_index]

        for j in self.interaction_indices:
            interaction = (
                mlogp * X[:, j]
            ).reshape(-1, 1)

            output.append(interaction)

        return np.hstack(output)

model_f04_nhm = Pipeline([
    (
        "imputer",
        IterativeImputer(
            random_state=RANDOM_STATE
        )
    ),

    (
        "features",
        SplinePlusSelectedInteractions(
            n_knots=6,
            degree=3,
            mlogp_index=4,
            interaction_indices=[0, 8]
        )
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        Ridge(alpha=100)
    )
])

fold_results, summary_f04_nhm = evaluate_model(
    model_f04_nhm,
    X,
    y,
    cv_reg
)

summary_f04_nhm

{'MAE_mean': np.float64(0.5949677563196047),
 'MAE_std': np.float64(0.054988095074217494),
 'RMSE_mean': np.float64(0.7696316999431501),
 'RMSE_std': np.float64(0.08508469212254684),
 'R2_mean': np.float64(0.6544931494903669),
 'R2_std': np.float64(0.09080887023632447)}

The best by now is model spline features for all 9 variables + MLOGP×F04[C-O] + MLOGP×nHM + Ridge(alpha=100),          ( (n_knots=6, degree=3)

Try to add B02[C-N] as well

In [39]:
model_f04_nhm_b02 = Pipeline([
    (
        "imputer",
        IterativeImputer(
            random_state=RANDOM_STATE
        )
    ),

    (
        "features",
        SplinePlusSelectedInteractions(
            n_knots=6,
            degree=3,
            mlogp_index=4,
            interaction_indices=[0, 7, 8]
        )
    ),

    (
        "scaler",
        StandardScaler()
    ),

    (
        "model",
        Ridge(alpha=100)
    )
])

fold_results_b02, summary_b02 = evaluate_model(
    model_f04_nhm_b02,
    X,
    y,
    cv_reg
)

summary_b02

{'MAE_mean': np.float64(0.5940489901414756),
 'MAE_std': np.float64(0.054989259548196015),
 'RMSE_mean': np.float64(0.769078102694966),
 'RMSE_std': np.float64(0.08578586694503161),
 'R2_mean': np.float64(0.6549589664612183),
 'R2_std': np.float64(0.09099595074050211)}

In [40]:
model_f04_nhm_b02.fit(X, y)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('imputer', ...), ('features', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](9,)","['nHM','piPC09','PCD',...,'N-072','B02[C-N]','F04[C-O]']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,9
,"random_state random_state: int, RandomState instance or None, default=NoneThe seed of the pseudo random number generator to use. Randomizesselection of estimator features if `n_nearest_features` is not `None`,the `imputation_order` if `random`, and the sampling from posterior if`sample_posterior=True`. Use an integer for determinism.See :term:`the Glossary <random_state>`.",51026
,"estimator estimator: estimator object, default=BayesianRidge()The estimator to use at each step of the round-robin imputation.If `sample_posterior=True`, the estimator must support`return_std` in its `predict` method.",None
,"missing_values missing_values: int or np.nan, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`should be set to `np.nan`, since `pd.NA` will be converted to `np.nan`.",nan
,"sample_posterior sample_posterior: bool, default=FalseWhether to sample from the (Gaussian) predictive posterior of thefitted estimator for each imputation. Estimator must support`return_std` in its `predict` method if set to `True`. Set to`True` if using `IterativeImputer` for multiple imputations.",False


In [42]:
final_predictions = model_f04_nhm_b02.predict(
    X_submission
).ravel()

final_predictions[:10]

array([0.83797502, 3.83110302, 0.72440206, 0.49741175, 0.70823296,
       1.37326743, 3.26057283, 1.82760894, 1.55290191, 1.66791101])

In [44]:
submission = pd.read_csv('submission1.csv')
submission.head(3)

,sample_id,regression_prediction,classification_prediction
0,78-79-5,0.861837,1
1,16606-02-3,3.797140,1
2,111-44-4,0.776910,1


In [47]:
# 1. Assign the predictions array to the DataFrame column
submission['regression_prediction'] = final_predictions

# 2. Verify the changes
print(submission.head(3))

# 3. Save the updated DataFrame to a CSV file
submission.to_csv('submission2.csv', index=False)

    sample_id  regression_prediction  classification_prediction
0     78-79-5               0.837975                          1
1  16606-02-3               3.831103                          1
2    111-44-4               0.724402                          1
